# Onboarding modernized filing stats spreadsheet notebook

## Overview
This notebook generates the data for the onboarding modernized filing stats spreadsheet.

## What it does
- Extracts migration data from COLIN Extract database
- Retrieves filing information from LEAR database
- Merges and exports data to Excel format

## Output
A formatted Excel spreadsheet for filing stats of each comapny.

In [ ]:
%pip install pandas
%pip install sqlalchemy>=2.0
%pip install oracledb
%pip install dotenv
%pip install psycopg2-binary
%pip install openpyxl

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError, OperationalError
from dotenv import load_dotenv
from datetime import datetime

load_dotenv()

COLUMN_NAMES = {
    "corp_num": "Incorporation Number",
    "corp_name": "Company Name",
    "filings": "Filings",
    "filing_date": "Filing Date"
}

CONFIG = {
    'excel_export': {
        'font_size': 14,
        'max_column_width': 65,
        'filled_color': 'FFCCCC',
        'output_dir': os.getenv('EXPORT_OUTPUT_DIR')
    }
}

In [ ]:
DATABASE_CONFIG = {
    'lear': {
        'username': os.getenv("DATABASE_LEAR_USERNAME"),
        'password': os.getenv("DATABASE_LEAR_PASSWORD"),
        'host': os.getenv("DATABASE_LEAR_HOST"),
        'port': os.getenv("DATABASE_LEAR_PORT"),
        'name': os.getenv("DATABASE_LEAR_NAME")
    }
}


for db_key, db_config in DATABASE_CONFIG.items():
    # Build PostgreSQL URI
    uri = f"postgresql://{db_config['username']}:{db_config['password']}@{db_config['host']}:{db_config['port']}/{db_config['name']}"
    DATABASE_CONFIG[db_key] = {'uri': uri}

print("Database configurations successfully.")

In [ ]:
engines = {}

for db_key, config in DATABASE_CONFIG.items():
    try:
        engine = create_engine(config['uri'])

        # Test connection
        with engine.connect() as conn:
            if db_key =='colin_oracle':
                conn.execute(text("SELECT 1 FROM DUAL"))
            else:
                conn.execute(text("SELECT 1"))

        engines[db_key] = engine
        print(f"{db_key.upper()} database engine created and tested successfully.")

    except OperationalError as e:
        print(f"{db_key.upper()} database connection failed: {e}")
        raise
    except SQLAlchemyError as e:
        print(f"{db_key.upper()} database engine creation failed: {e}")
        raise
    except Exception as e:
        print(f"{db_key.upper()} unexpected error: {e}")
        raise

ENGINE_NAMES = {engine: key for key, engine in engines.items()}

print("All database engines ready for use.")

In [ ]:
corp_type_codes = ('BC', 'C', 'ULC', 'CUL', 'CC', 'CCC')

lear_identifiers_query = f"""
SELECT DISTINCT
    b.identifier
FROM businesses b
JOIN filings f ON b.id = f.business_id
WHERE
    f.status = 'TOMBSTONE'
    AND b.legal_type IN {corp_type_codes}
ORDER BY
    b.identifier
"""

try:
    with engines['lear'].connect() as conn:
        lear_df = pd.read_sql(lear_identifiers_query, conn)
        print(f"Fetched {len(lear_df)} rows from LEAR database.")
        # cretae list of corp nums for LEAR query
        corp_nums = lear_df['identifier'].tolist()
        corp_nums_str = ','.join(f"'{num}'" for num in corp_nums)

except Exception as e:
    print(f"Error fetching data from LEAR: {e}")
    raise

# Display results
with pd.option_context('display.max_rows', None):
    display(lear_df.head(5))


In [ ]:
lear_query = f"""
SELECT
    b.identifier AS "{COLUMN_NAMES['corp_num']}",
    b.legal_name AS "{COLUMN_NAMES['corp_name']}",
    
    COALESCE(
        STRING_AGG(f.filing_type, ', '), ''
    ) AS "{COLUMN_NAMES['filings']}",
    COALESCE(
        STRING_AGG(f.filing_date::text, ', '), ''
    ) AS "{COLUMN_NAMES['filing_date']}"
FROM businesses b
LEFT JOIN filings f ON b.id = f.business_id
    AND f.source = 'LEAR'
    AND f.status = 'COMPLETED'
WHERE b.identifier IN ({corp_nums_str})
GROUP BY b.identifier, b.legal_name, f.filing_type, f.filing_date
"""

try:
    with engines['lear'].connect() as conn:
        lear_df = pd.read_sql(lear_query, conn)
        print(f"Fetched {len(lear_df)} rows from LEAR database.")

except Exception as e:
    print(f"Error fetching data from LEAR: {e}")
    raise

# Display results
with pd.option_context('display.max_rows', None):
    display(lear_df.head(5))

# Create a new excel file with the dataframe
output_file = os.path.join(CONFIG['excel_export']['output_dir'], f"lear_filing_stats_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx")

try:
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        lear_df.to_excel(writer, sheet_name='Sheet1', index=False)
        workbook = writer.book
        worksheet = writer.sheets['Sheet1']
        
        # Adjust column widths for better visibility
        for column in worksheet.columns:
            max_length = 0
            column_letter = column[0].column_letter
            for cell in column:
                try:
                    if len(str(cell.value)) > max_length:
                        max_length = len(str(cell.value))
                except:
                    pass
            # Add extra padding and cap at max width
            adjusted_width = min(max_length + 3, CONFIG['excel_export']['max_column_width'])
            worksheet.column_dimensions[column_letter].width = adjusted_width
        
        # Enable text wrapping for better content visibility
        from openpyxl.styles import Alignment
        for row in worksheet.iter_rows(min_row=1, max_row=worksheet.max_row, min_col=1, max_col=worksheet.max_column):
            for cell in row:
                cell.alignment = Alignment(wrap_text=True, vertical='top')
    
    print(f"Data exported to Excel file: {output_file}")
except Exception as e:
    print(f"Error exporting data to Excel: {e}")
    raise